# AIC 2026 — Speech-to-Text với VinAI PhoWhisper Large

Notebook quét video, chạy model VinAI PhoWhisper Large, rồi lưu JSON, SRT và TXT. File đã hoàn thành sẽ được bỏ qua để có thể tiếp tục khi runtime ngắt kết nối.

Notebook chạy được trên **cả Google Colab và Kaggle**, tự phát hiện môi trường:

| | Colab | Kaggle |
|---|---|---|
| Video | `MyDrive/AI Challenge/Dataset_Directory` | `/kaggle/input/datasets/fatle542/aic-dataset` |
| Transcript | `MyDrive/AI Challenge/Transcripts` | `/kaggle/working/Transcripts` |

**Colab**: Runtime → Change runtime type → GPU (A100/L4 khuyến nghị).

**Kaggle**: Settings → Accelerator → **GPU T4 x2 / P100**, và bật **Internet: On** (cần tải model từ Hugging Face). Lưu ý `/kaggle/input` là read-only nên kết quả ghi vào `/kaggle/working`; nhớ tải về hoặc *Save Version* trước khi hết session, vì `/kaggle/working` bị xoá khi session kết thúc.

In [1]:
import os
import shutil
import subprocess
import sys

def detect_env():
    """Phát hiện môi trường: 'kaggle' | 'colab' | 'local'.

    Kiểm tra Kaggle TRƯỚC vì /kaggle là dấu hiệu chắc chắn; image của Kaggle có
    thể khiến các tín hiệu của Colab khớp sai.
    """
    if os.path.isdir('/kaggle/input') or os.path.isdir('/kaggle/working') or os.environ.get('KAGGLE_KERNEL_RUN_TYPE'):
        return 'kaggle'
    if os.environ.get('COLAB_RELEASE_TAG') or os.path.isdir('/content') or 'google.colab' in sys.modules:
        return 'colab'
    return 'local'

ENV = detect_env()
print('Môi trường:', ENV)
print('  /kaggle/input:', os.path.isdir('/kaggle/input'),
      '| KAGGLE_KERNEL_RUN_TYPE:', os.environ.get('KAGGLE_KERNEL_RUN_TYPE'),
      '| /content:', os.path.isdir('/content'),
      '| COLAB_RELEASE_TAG:', os.environ.get('COLAB_RELEASE_TAG'))

subprocess.run(['nvidia-smi'], check=False)
subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', '-U', 'transformers', 'accelerate'], check=True)

# Kaggle đã có ffmpeg sẵn; chỉ cài khi thiếu.
if shutil.which('ffmpeg') is None:
    subprocess.run('apt-get -qq update && apt-get -qq install -y ffmpeg', shell=True, check=True)
print('ffmpeg:', shutil.which('ffmpeg'))

Môi trường: kaggle
  /kaggle/input: True | KAGGLE_KERNEL_RUN_TYPE: Batch | /content: True | COLAB_RELEASE_TAG: release-colab-external-images_20260514-060047_RC00
Tue Aug 11 07:55:58 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8              9W /   70W |       0MiB /  15360MiB |  

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 79.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 35.0 MB/s eta 0:00:00


ffmpeg: /usr/bin/ffmpeg


In [2]:
# Chỉ Colab cần mount Drive. Trên Kaggle dataset đã có sẵn ở /kaggle/input.
if ENV == 'colab':
    from google.colab import drive
    drive.mount('/content/drive')
else:
    print('Bỏ qua mount Drive (ENV =', ENV, ')')

Bỏ qua mount Drive (ENV = kaggle )


## Cấu hình

Đường dẫn được chọn tự động theo môi trường:

- **Colab**: đọc `AI Challenge/Dataset_Directory`, ghi vào `AI Challenge/Transcripts` trên Drive.
- **Kaggle**: đọc `/kaggle/input/datasets/fatle542/aic-dataset` (read-only), ghi vào `/kaggle/working/Transcripts`.

`TARGET_FOLDERS` là danh sách thư mục video cần xử lý. Để `TARGET_FOLDERS = ['.']` nếu muốn quét toàn bộ dataset. Cell sẽ in ra các thư mục thực có để bạn đối chiếu (cấu trúc dataset trên Kaggle có thể lồng thêm một cấp).

In [3]:
import os
import sys
import zipfile
from pathlib import Path

# ENV được đặt ở cell đầu; tính lại nếu kernel vừa restart.
# Muốn ép môi trường thì gán thẳng ở đây, ví dụ: ENV = 'kaggle'
try:
    ENV
except NameError:
    if os.path.isdir('/kaggle/input') or os.path.isdir('/kaggle/working') or os.environ.get('KAGGLE_KERNEL_RUN_TYPE'):
        ENV = 'kaggle'
    elif os.environ.get('COLAB_RELEASE_TAG') or os.path.isdir('/content'):
        ENV = 'colab'
    else:
        ENV = 'local'
print('Môi trường:', ENV)

#@title Đường dẫn theo môi trường
if ENV == 'kaggle':
    # Kaggle: /kaggle/input là read-only nên transcript phải ghi ra /kaggle/working.
    KAGGLE_DATASET_CANDIDATES = [
        Path('/kaggle/input/datasets/fatle542/aic-dataset'),
        Path('/kaggle/input/aic-dataset'),
    ]
    DATASET_DIRECTORY = next((p for p in KAGGLE_DATASET_CANDIDATES if p.is_dir()), KAGGLE_DATASET_CANDIDATES[0])
    TRANSCRIPTS_DIRECTORY = Path('/kaggle/working/Transcripts')
elif ENV == 'colab':
    DATASET_DIRECTORY = Path('/content/drive/MyDrive/AI Challenge/Dataset_Directory')
    TRANSCRIPTS_DIRECTORY = Path('/content/drive/MyDrive/AI Challenge/Transcripts')
else:
    DATASET_DIRECTORY = Path('./Dataset_Directory')
    TRANSCRIPTS_DIRECTORY = Path('./Transcripts')

assert DATASET_DIRECTORY.is_dir(), (
    f'Không tìm thấy dataset: {DATASET_DIRECTORY}\n'
    + ('Kiểm tra tên dataset đã Add vào notebook. Có trong /kaggle/input: '
       + ', '.join(sorted(p.name for p in Path('/kaggle/input').iterdir())) if ENV == 'kaggle' else '')
)

# Chỉ giữ lại các thư mục bạn muốn chuyển giọng nói thành văn bản trong mảng này.
# Dùng ['.'] để quét toàn bộ dataset.
TARGET_FOLDERS = [
      'Videos_L21_a',
      'Videos_L22_a',
      'Videos_L23_a',
      'Videos_L24_a',
      'Videos_L25_a',
      'Videos_L26_a',
      'Videos_L26_b',
]

# Bắt đầu từ video này trong danh sách đã sắp xếp, bỏ qua mọi video đứng trước nó.
# Dùng khi session Kaggle trước hết giờ: điền tên video đang chạy dở, ví dụ 'L25_V043'.
# Đặt None để quét từ đầu (video đã có transcript vẫn tự động SKIP).
RESUME_FROM_VIDEO = 'L25_V043'

MODEL_ID = 'vinai/PhoWhisper-large'
OVERWRITE = False
VIDEO_EXTENSIONS = {'.mp4', '.mkv', '.mov', '.avi', '.webm', '.m4v', '.mpeg', '.mpg'}

dataset_root_resolved = DATASET_DIRECTORY.resolve()

# Liệt kê các thư mục con thực có (2 cấp) để đối chiếu — cấu trúc trên Kaggle có thể lồng thêm một cấp.
AVAILABLE_VIDEO_FOLDERS = sorted(
    str(p.relative_to(dataset_root_resolved))
    for depth in ('*', '*/*')
    for p in dataset_root_resolved.glob(depth)
    if p.is_dir()
)
print('Dataset:', dataset_root_resolved)
print('Thư mục con hiện có:')
for folder in AVAILABLE_VIDEO_FOLDERS[:40]:
    print('  -', folder)
if len(AVAILABLE_VIDEO_FOLDERS) > 40:
    print(f'  ... và {len(AVAILABLE_VIDEO_FOLDERS) - 40} thư mục khác')

def resolve_target(folder):
    """Trả về đường dẫn tuyệt đối của thư mục video, tìm cả ở cấp lồng bên trong."""
    relative_folder = Path(str(folder).strip() or '.')
    assert not relative_folder.is_absolute(), f'Thư mục phải là đường dẫn tương đối: {folder}'
    candidate = (DATASET_DIRECTORY / relative_folder).resolve()
    if not candidate.is_dir():
        # Dataset trên Kaggle thường bọc thêm một thư mục gốc → tìm theo tên.
        matches = [p for p in dataset_root_resolved.glob(f'*/{relative_folder}') if p.is_dir()]
        assert matches, (
            f'Không tìm thấy thư mục: {folder}\n'
            f'Các thư mục hiện có: {AVAILABLE_VIDEO_FOLDERS[:40]}'
        )
        candidate = matches[0].resolve()
    assert candidate == dataset_root_resolved or dataset_root_resolved in candidate.parents, \
        f'Thư mục không được nằm ngoài dataset: {folder}'
    return candidate

assert TARGET_FOLDERS, 'TARGET_FOLDERS không được để trống'
VIDEO_ROOTS = [resolve_target(folder) for folder in TARGET_FOLDERS]

VIDEO_ROOT = dataset_root_resolved  # Thư mục gốc chung để giữ cấu trúc khi lưu kết quả
TRANSCRIPTS_DIRECTORY.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT = TRANSCRIPTS_DIRECTORY.resolve()

# --- Resume: nhận diện transcript của các lần chạy trước ---
# Kaggle xoá /kaggle/working khi session kết thúc. Sau mỗi lần chạy hãy bấm
# Save Version, rồi Add Data notebook output đó vào lần chạy sau — transcript
# cũ sẽ được nhận ra ở đây và tự động bỏ qua (khớp theo tên video, ví dụ L25_V043).
RESUME_ROOTS = []
if ENV == 'kaggle' and Path('/kaggle/input').is_dir():
    dataset_branch = {dataset_root_resolved, *dataset_root_resolved.parents}
    RESUME_ROOTS = [
        p for p in sorted(Path('/kaggle/input').iterdir())
        if p.is_dir() and p.resolve() not in dataset_branch
    ]
# Transcript còn sót trong thư mục output hiện tại cũng được tính.
RESUME_ROOTS.append(OUTPUT_ROOT)

# Cell đóng gói ở cuối notebook tạo transcripts.zip; giải nén lại vào OUTPUT_ROOT
# (zip được tạo với root_dir = OUTPUT_ROOT nên cấu trúc thư mục khớp sẵn).
ZIP_SEARCH_ROOTS = list(RESUME_ROOTS)
if ENV == 'kaggle' and Path('/kaggle/working').is_dir():
    ZIP_SEARCH_ROOTS.append(Path('/kaggle/working'))

seen_zips = set()
for zip_root in ZIP_SEARCH_ROOTS:
    for zip_path in sorted(zip_root.rglob('*.zip')):
        resolved_zip = zip_path.resolve()
        if resolved_zip in seen_zips:
            continue
        seen_zips.add(resolved_zip)
        try:
            with zipfile.ZipFile(zip_path) as archive:
                members = [
                    name for name in archive.namelist()
                    if name.endswith(('.json', '.txt', '.srt'))
                ]
                if not members:
                    continue
                archive.extractall(OUTPUT_ROOT, members=members)
        except zipfile.BadZipFile:
            print('Bỏ qua (zip hỏng):', zip_path)
            continue
        print(f'Đã giải nén {len(members)} file từ {zip_path} -> {OUTPUT_ROOT}')

PREVIOUS_TRANSCRIPTS = {}
for resume_root in RESUME_ROOTS:
    for json_path in resume_root.rglob('*.json'):
        if json_path.name.startswith('_'):
            continue
        PREVIOUS_TRANSCRIPTS.setdefault(json_path.stem, json_path)

print('\nResume — quét transcript cũ ở:')
for resume_root in RESUME_ROOTS:
    print('-', resume_root)
print(f'Tìm thấy {len(PREVIOUS_TRANSCRIPTS)} transcript đã có sẵn')

print('\nCác thư mục đầu vào:')
for root in VIDEO_ROOTS:
    print('-', root)
print('Output:', OUTPUT_ROOT)

Môi trường: kaggle
Dataset: /kaggle/input/datasets/fatle542/aic-dataset
Thư mục con hiện có:
  - Keyframes_L21
  - Keyframes_L21/keyframes
  - Keyframes_L22
  - Keyframes_L22/keyframes
  - Keyframes_L23
  - Keyframes_L23/keyframes
  - Keyframes_L24
  - Keyframes_L24/keyframes
  - Keyframes_L25
  - Keyframes_L25/keyframes
  - Keyframes_L26_a
  - Keyframes_L26_a/keyframes
  - Keyframes_L26_b
  - Keyframes_L26_b/keyframes
  - Keyframes_L26_c
  - Keyframes_L26_c/keyframes
  - Keyframes_L26_d
  - Keyframes_L26_d/keyframes
  - Keyframes_L26_e
  - Keyframes_L26_e/keyframes
  - Keyframes_L27
  - Keyframes_L27/keyframes
  - Keyframes_L28
  - Keyframes_L28/keyframes
  - Keyframes_L29
  - Keyframes_L29/keyframes
  - Keyframes_L30
  - Keyframes_L30/keyframes
  - Videos_L21_a
  - Videos_L21_a/video
  - Videos_L22_a
  - Videos_L22_a/video
  - Videos_L23_a
  - Videos_L23_a/video
  - Videos_L24_a
  - Videos_L24_a/video
  - Videos_L25_a
  - Videos_L25_a/video
  - Videos_L26_a
  - Videos_L26_a/video
  .

In [4]:
import torch
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline

assert torch.cuda.is_available(), (
    'Chưa bật GPU: Kaggle → Settings → Accelerator → GPU T4 x2 / P100'
    if ENV == 'kaggle' else
    'Chưa bật GPU: Runtime → Change runtime type → GPU'
)

# Mỗi GPU giữ một bản model riêng và xử lý một video tại một thời điểm.
# Nếu chỉ có một GPU, notebook tự động quay về chế độ single-GPU.
GPU_COUNT = min(2, torch.cuda.device_count())
model_dtype = torch.float16
print(f'Phát hiện {torch.cuda.device_count()} GPU; sẽ dùng {GPU_COUNT} GPU:')
for gpu_id in range(GPU_COUNT):
    print(f'  cuda:{gpu_id}: {torch.cuda.get_device_name(gpu_id)}')

def build_transcriber(gpu_id):
    device = f'cuda:{gpu_id}'
    print(f'[GPU {gpu_id}] Đang tải {MODEL_ID}...')
    processor = AutoProcessor.from_pretrained(MODEL_ID)
    model = AutoModelForSpeechSeq2Seq.from_pretrained(
        MODEL_ID, dtype=model_dtype, low_cpu_mem_usage=True,
    ).to(device)
    asr = pipeline(
        'automatic-speech-recognition',
        model=model,
        tokenizer=processor.tokenizer,
        feature_extractor=processor.feature_extractor,
        dtype=model_dtype,
        device=device,
    )
    print(f'[GPU {gpu_id}] Đã tải model')
    return asr

transcribers = [build_transcriber(gpu_id) for gpu_id in range(GPU_COUNT)]
print(f'Sẵn sàng chạy với {GPU_COUNT} GPU')

Phát hiện 2 GPU; sẽ dùng 2 GPU:
  cuda:0: Tesla T4
  cuda:1: Tesla T4
[GPU 0] Đang tải vinai/PhoWhisper-large...


preprocessor_config.json:   0%|          | 0.00/339 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/805 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/1260 [00:00<?, ?it/s]

generation_config.json: 0.00B [00:00, ?B/s]

[GPU 0] Đã tải model
[GPU 1] Đang tải vinai/PhoWhisper-large...


Loading weights:   0%|          | 0/1260 [00:00<?, ?it/s]

[GPU 1] Đã tải model
Sẵn sàng chạy với 2 GPU


In [5]:
videos = sorted(
    {
        p
        for root in VIDEO_ROOTS
        for p in root.rglob('*')
        if p.is_file() and p.suffix.lower() in VIDEO_EXTENSIONS
    },
    key=lambda p: str(p).lower(),
)
print(f'Tìm thấy {len(videos)} video')

if RESUME_FROM_VIDEO:
    wanted_stem = Path(RESUME_FROM_VIDEO).stem
    start_index = next((i for i, p in enumerate(videos) if p.stem == wanted_stem), None)
    assert start_index is not None, (
        f'Không tìm thấy RESUME_FROM_VIDEO={RESUME_FROM_VIDEO!r} trong danh sách. '
        'Kiểm tra lại TARGET_FOLDERS hoặc đặt RESUME_FROM_VIDEO = None.'
    )
    videos = videos[start_index:]
    print(
        f'RESUME_FROM_VIDEO={wanted_stem}: bỏ qua {start_index} video đứng trước, '
        f'còn {len(videos)} video sẽ xử lý (đánh số [1/{len(videos)}] tính lại từ đây)'
    )
for video in videos[:10]:
    print('-', video.relative_to(VIDEO_ROOT))

Tìm thấy 415 video
- Videos_L21_a/video/L21_V001.mp4
- Videos_L21_a/video/L21_V002.mp4
- Videos_L21_a/video/L21_V003.mp4
- Videos_L21_a/video/L21_V005.mp4
- Videos_L21_a/video/L21_V006.mp4
- Videos_L21_a/video/L21_V007.mp4
- Videos_L21_a/video/L21_V008.mp4
- Videos_L21_a/video/L21_V009.mp4
- Videos_L21_a/video/L21_V010.mp4
- Videos_L21_a/video/L21_V011.mp4


In [6]:
import json
import subprocess
import traceback

import numpy as np

SAMPLE_RATE = 16_000

def load_audio(path, sampling_rate=SAMPLE_RATE):
    """Giải mã audio thành mảng float32 mono bằng ffmpeg.

    Không truyền đường dẫn trực tiếp cho pipeline: transformers sẽ đọc cả file
    thành bytes rồi bơm qua stdin của ffmpeg, mà MP4 cần input seekable nên trên
    một số bản ffmpeg (Kaggle) sẽ trả về 0 byte. Đọc thẳng từ file thì luôn ổn.
    """
    command = [
        'ffmpeg', '-nostdin', '-threads', '0',
        '-i', str(path),
        '-vn', '-f', 'f32le', '-acodec', 'pcm_f32le',
        '-ac', '1', '-ar', str(sampling_rate),
        '-loglevel', 'error', '-',
    ]
    process = subprocess.run(command, capture_output=True)
    if process.returncode != 0:
        stderr = process.stderr.decode('utf-8', 'ignore').strip()
        raise RuntimeError(f'ffmpeg lỗi khi đọc {path}:\n{stderr[-2000:]}')
    audio = np.frombuffer(process.stdout, dtype=np.float32)
    if audio.size == 0:
        raise RuntimeError(f'Không giải mã được audio (video không có tiếng?): {path}')
    return {'raw': audio.copy(), 'sampling_rate': sampling_rate}

def transcribe(asr, video):
    return asr(
        load_audio(video),
        return_timestamps=True,
        generate_kwargs={'language': 'vi', 'task': 'transcribe'},
    )

def srt_timestamp(seconds):
    milliseconds = max(0, round(float(seconds) * 1000))
    hours, remainder = divmod(milliseconds, 3_600_000)
    minutes, remainder = divmod(remainder, 60_000)
    secs, millis = divmod(remainder, 1000)
    return f'{hours:02d}:{minutes:02d}:{secs:02d},{millis:03d}'

def format_timestamp(seconds):
    if seconds is None:
        return '??:??:??.???'
    return srt_timestamp(seconds).replace(',', '.')

def save_transcript(video, result):
    relative = video.relative_to(VIDEO_ROOT).with_suffix('')
    stem = OUTPUT_ROOT / relative
    stem.parent.mkdir(parents=True, exist_ok=True)
    segments = []
    for index, chunk in enumerate(result.get('chunks', [])):
        timestamp = chunk.get('timestamp') or (0.0, 0.0)
        start = float(timestamp[0] or 0.0)
        end = float(timestamp[1] if timestamp[1] is not None else start)
        segments.append({
            'id': index,
            'start': start,
            'end': end,
            'video_start': start,
            'video_end': end,
            'text': chunk.get('text', '').strip(),
        })
    payload = {
        'source': str(video),
        'video_id': video.stem,
        'model': MODEL_ID,
        'language': 'vi',
        'text': result.get('text', '').strip(),
        'segments': segments,
    }
    stem.with_suffix('.json').write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')
    timestamped_text = '\n'.join(
        f"[{format_timestamp(segment['video_start'])} --> {format_timestamp(segment['video_end'])}] {segment['text']}"
        for segment in segments
    )
    stem.with_suffix('.txt').write_text(
        (timestamped_text or payload['text']) + '\n', encoding='utf-8'
    )
    blocks = []
    for index, segment in enumerate(payload['segments'], 1):
        text = segment.get('text', '').strip()
        blocks.append(f"{index}\n{srt_timestamp(segment['start'])} --> {srt_timestamp(segment['end'])}\n{text}")
    stem.with_suffix('.srt').write_text('\n\n'.join(blocks) + ('\n' if blocks else ''), encoding='utf-8')
    return stem

def output_json_path(video):
    return (OUTPUT_ROOT / video.relative_to(VIDEO_ROOT)).with_suffix('.json')

## Chạy thử một video

Nên chạy cell này trước để xác nhận đường dẫn, GPU và chất lượng transcript.

In [7]:
assert videos, 'Không tìm thấy video nào'
sample_video = videos[0]

# Kiểm tra khâu giải mã audio trước khi chạy model — lỗi ffmpeg sẽ lộ ra ngay ở đây.
sample_audio = load_audio(sample_video)
print('Video:', sample_video)
print(f"Audio: {sample_audio['raw'].size / SAMPLE_RATE:.1f}s @ {sample_audio['sampling_rate']} Hz")

sample_result = transcribe(transcribers[0], sample_video)
sample_stem = save_transcript(sample_video, sample_result)
print('Đã lưu:', sample_stem)
print('\n--- KẾT QUẢ CÓ TIMESTAMP TRONG VIDEO ---\n')
for chunk in sample_result.get('chunks', []):
    start, end = chunk.get('timestamp') or (None, None)
    print(f"[{format_timestamp(start)} --> {format_timestamp(end)}] {chunk.get('text', '').strip()}")
if not sample_result.get('chunks'):
    print(sample_result.get('text', '')[:1000])

Video: /kaggle/input/datasets/fatle542/aic-dataset/Videos_L21_a/video/L21_V001.mp4
Audio: 1261.7s @ 16000 Hz


[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.


[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> to see related `.generate()` flags.


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Đã lưu: /kaggle/working/Transcripts/Videos_L21_a/video/L21_V001

--- KẾT QUẢ CÓ TIMESTAMP TRONG VIDEO ---

[00:00:00.020 --> 00:00:28.020] chào mừng quý vị đến với chương trình sáu mươi giây của đài truyền hình thành phố hồ chí minh chương trình sách này có những thông tin nổi bật sau đây đồng băng sông cửu long với tình trạng sụt lún gấp gần hai mươi lần so với nước biển dần vẫn chuyển các tốp trái tim từ hà nội về huế ghép cho bệnh nhân châu âu trúng chọi với nhiệt độ nóng như thiêu đốt cùng những đám cháy rực.
[00:00:30.020 --> 00:00:59.860] sụt lún đang là vấn đề cấp bách với đồng bằng sông cửu long khi có nơi sụt lún trung bình lên tới năm phẩy bảy xăngtimét một năm tức là gấp gần hai mươi lần so với nước biển dần dự báo phần lớn diện tích có thể sẽ nằm dưới mực ờ nước biển trung bình vào cuối thế kỷ hai mươi mốt chiều ngày ba mươi mốt tháng bảy tại hà nội cơ quan chuyên môn đã báo cáo lãnh đạo bộ nông nghiệp và phát triển nông thôn về đề án tổng thể phòng chống sụt lún đất sạt lở

## Chạy toàn bộ video

Cell này tự bỏ qua các video đã có `.json` khớp `MODEL_ID`, nên **chạy lại là tiếp tục từ chỗ dừng**. Nếu runtime ngắt kết nối, chỉ cần chạy lại các cell setup (mount Drive nếu ở Colab, load model) rồi chạy lại cell này.

### Tiếp tục trên Kaggle sau khi session kết thúc

`/kaggle/working` bị xoá khi session dừng, nên phải mang kết quả cũ quay lại:

1. Ở lần chạy trước, bấm **Save Version** (Save & Run All hoặc Quick Save) để output được giữ lại.
2. Lần chạy mới: **Add Data → Your Work / Notebook Output**, chọn output của version đó.
3. Chạy lại từ đầu. Cell cấu hình sẽ in `Tìm thấy N transcript đã có sẵn`; những video đó được chép sang `/kaggle/working/Transcripts` và in `SKIP`, phần còn lại chạy tiếp.

Vì session Kaggle chỉ ~9–12 giờ, nên chia `TARGET_FOLDERS` thành từng đợt (ví dụ 2–3 thư mục `Videos_L2x_a` mỗi lần) thay vì chạy cả 415 video trong một lần.


In [8]:
import shutil
from concurrent.futures import ThreadPoolExecutor
from queue import Queue, Empty
from threading import Lock

jobs = Queue()
for index, video in enumerate(videos, 1):
    jobs.put((index, video))

print_lock = Lock()

def output_is_complete(video):
    json_path = output_json_path(video)
    stem = json_path.with_suffix('')
    if not all(stem.with_suffix(ext).exists() for ext in ('.json', '.txt', '.srt')):
        return False, None
    try:
        existing_model = json.loads(json_path.read_text(encoding='utf-8')).get('model')
    except Exception:
        return False, None
    return existing_model == MODEL_ID, existing_model

def restore_previous(video):
    """Chép transcript của lần chạy trước vào OUTPUT_ROOT để lần này bỏ qua video đó."""
    source_json = PREVIOUS_TRANSCRIPTS.get(video.stem)
    if source_json is None:
        return False
    destination_stem = output_json_path(video).with_suffix('')
    if source_json.resolve() == destination_stem.with_suffix('.json').resolve():
        return False
    destination_stem.parent.mkdir(parents=True, exist_ok=True)
    restored = False
    for extension in ('.json', '.txt', '.srt'):
        source = source_json.with_suffix(extension)
        if source.exists():
            shutil.copy2(source, destination_stem.with_suffix(extension))
            restored = True
    return restored

def gpu_worker(gpu_id):
    asr = transcribers[gpu_id]
    local_success = local_skipped = local_failed = 0
    local_failures = []
    with torch.cuda.device(gpu_id):
        while True:
            try:
                index, video = jobs.get_nowait()
            except Empty:
                break
            try:
                complete, existing_model = output_is_complete(video)
                if not complete and not OVERWRITE and restore_previous(video):
                    complete, existing_model = output_is_complete(video)
                if complete and not OVERWRITE:
                    local_skipped += 1
                    with print_lock:
                        print(f'[GPU {gpu_id}] [{index}/{len(videos)}] SKIP {video.name}')
                    continue
                if existing_model is not None and not OVERWRITE:
                    with print_lock:
                        print(f'[GPU {gpu_id}] [{index}/{len(videos)}] REPROCESS {video.name} (model cũ: {existing_model})')
                with print_lock:
                    print(f'[GPU {gpu_id}] [{index}/{len(videos)}] STT  {video}')
                result = transcribe(asr, video)
                save_transcript(video, result)
                local_success += 1
            except Exception as error:
                local_failed += 1
                local_failures.append({'video': str(video), 'gpu': gpu_id, 'error': repr(error)})
                with print_lock:
                    print(f'[GPU {gpu_id}] ERROR {video}: {error!r}')
                    print(traceback.format_exc())
            finally:
                jobs.task_done()
                torch.cuda.empty_cache()
    return local_success, local_skipped, local_failed, local_failures

with ThreadPoolExecutor(max_workers=GPU_COUNT) as executor:
    worker_results = list(executor.map(gpu_worker, range(GPU_COUNT)))

success = sum(item[0] for item in worker_results)
skipped = sum(item[1] for item in worker_results)
failed = sum(item[2] for item in worker_results)
failed_videos = [failure for item in worker_results for failure in item[3]]
(OUTPUT_ROOT / '_failed.json').write_text(
    json.dumps(failed_videos, ensure_ascii=False, indent=2), encoding='utf-8'
)
print(f'Hoàn tất với {GPU_COUNT} GPU: success={success}, skipped={skipped}, failed={failed}, total={len(videos)}')

[GPU 1] [2/415] STT  /kaggle/input/datasets/fatle542/aic-dataset/Videos_L21_a/video/L21_V002.mp4
[GPU 0] [1/415] SKIP L21_V001.mp4
[GPU 0] [3/415] STT  /kaggle/input/datasets/fatle542/aic-dataset/Videos_L21_a/video/L21_V003.mp4


[transformers] Whisper did not predict an ending timestamp, which can happen if audio is cut off in the middle of a word. Also make sure WhisperTimeStampLogitsProcessor was used during generation.


[GPU 1] [4/415] STT  /kaggle/input/datasets/fatle542/aic-dataset/Videos_L21_a/video/L21_V005.mp4


[GPU 0] [5/415] STT  /kaggle/input/datasets/fatle542/aic-dataset/Videos_L21_a/video/L21_V006.mp4


[transformers] Whisper did not predict an ending timestamp, which can happen if audio is cut off in the middle of a word. Also make sure WhisperTimeStampLogitsProcessor was used during generation.


...

[GPU 0] [170/415] STT  /kaggle/input/datasets/fatle542/aic-dataset/Videos_L25_a/video/L25_V042.mp4


[transformers] Whisper did not predict an ending timestamp, which can happen if audio is cut off in the middle of a word. Also make sure WhisperTimeStampLogitsProcessor was used during generation.


[GPU 1] [171/415] STT  /kaggle/input/datasets/fatle542/aic-dataset/Videos_L25_a/video/L25_V043.mp4


[GPU 1] [172/415] STT  /kaggle/input/datasets/fatle542/aic-dataset/Videos_L25_a/video/L25_V044.mp4


[transformers] Whisper did not predict an ending timestamp, which can happen if audio is cut off in the middle of a word. Also make sure WhisperTimeStampLogitsProcessor was used during generation.


## Đóng gói kết quả (Kaggle)

Nén toàn bộ transcript thành một file zip trong `/kaggle/working` để tải về từ tab **Output**. Trên Colab kết quả đã nằm sẵn trên Drive nên cell này chỉ báo bỏ qua.

In [ ]:
if ENV == 'kaggle':
    import shutil

    archive = shutil.make_archive('/kaggle/working/transcripts', 'zip', root_dir=OUTPUT_ROOT)
    size_mb = Path(archive).stat().st_size / 1024 / 1024
    print(f'Đã đóng gói: {archive} ({size_mb:.1f} MB)')
    print('Tải về ở panel Output bên phải, hoặc Save Version để giữ lại.')
else:
    print('Bỏ qua: kết quả đã nằm ở', OUTPUT_ROOT)